## Toy spiking neurons

a network of 100 toy spiking neurons. each neuron is a thread on my SMP machine. there is a random, but frozen connectivity between neurons such that each are connected by a delay between 2ms and 40 ms. each neuron emits spikes as a poisson process at a rate of one spike per second and sends it to the neurons it is connected to. I wish to record the spikes received by each neuron.
 



In [ ]:
import numpy as np


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Very compact spiking‑neuron network

* 100 neurons (threads)
* Fixed random connectivity – each outgoing edge gets a delay
  uniformly drawn once from 2 ms … 40 ms.
* Each neuron fires as a Poisson process (mean 1 spike / s).
* When a spike is delivered the *receiver* only increments a counter.
* The whole simulation runs for SIM_TIME seconds and then prints the
  spike‑count for every neuron.
"""

import threading
import time
import random
import queue
from collections import defaultdict

# ----------------------------------------------------------------------
# Parameters (feel free to change)
# ----------------------------------------------------------------------
N_NEURONS           = 100               # number of neurons
AVG_OUT_DEGREE     = 10                # ~10 outgoing links per neuron
MIN_DELAY_MS        = 2                 # axonal delay limits
MAX_DELAY_MS        = 40
SPIKE_RATE_HZ      = 1.0               # 1 Hz Poisson firing
SIM_TIME           = 5.0               # seconds the wall‑clock runs

# ----------------------------------------------------------------------
# Global, thread‑safe data structures
# ----------------------------------------------------------------------
spike_counts       = defaultdict(int)   # neuron_id → how many spikes received
counts_lock        = threading.Lock()
stop_event         = threading.Event()   # tells every thread to quit

# ----------------------------------------------------------------------
# Helper: monotonic clock (high‑resolution, never goes backward)
# ----------------------------------------------------------------------
now = time.perf_counter          # alias – a tiny bit faster to call


class Neuron(threading.Thread):
    """
    One toy neuron.

    * `connections`  : list of (target_neuron_obj, delay_seconds)
    * `spike_queue` : PriorityQueue[delivery_time]  – only the time matters
    * `t_start`      : moment this thread started (monotonic)
    * `t_next_spike` : absolute time (relative to t_start) of the next Poisson spike
    """

    def __init__(self, nid, connections):
        super().__init__(daemon=True)          # daemon → exits with program
        self.nid         = nid
        self.connections = connections
        self.spike_queue = queue.PriorityQueue()   # (delivery_time,)
        self.t_start     = None
        self.t_next_spike = None

    # ------------------------------------------------------------------
    # Schedule the *next* Poisson spike (draw a fresh exponential interval)
    # ------------------------------------------------------------------
    def _schedule_next_spike(self):
        interval = random.expovariate(SPIKE_RATE_HZ)   # seconds
        self.t_next_spike = self.t_now + interval

    # ------------------------------------------------------------------
    # Main thread body
    # ------------------------------------------------------------------
    def run(self):
        # Reference moment for this neuron
        self.t_start = now()
        self.t_now   = 0.0                # = now() - self.t_start
        self._schedule_next_spike()       # initialise first spike time

        while not stop_event.is_set():
            # ---- update local clock -------------------------------------------------
            self.t_now = now() - self.t_start

            # ---- Poisson spike generation -------------------------------------------
            if self.t_now >= self.t_next_spike:
                # fire to every post‑synaptic partner
                for tgt, delay_s in self.connections:
                    delivery = self.t_now + delay_s
                    tgt.spike_queue.put((delivery,))      # only the timestamp matters
                self._schedule_next_spike()

            # ---- process incoming spikes that are due -------------------------------
            try:
                while True:                                   # drain everything ready now
                    (delivery_time,) = self.spike_queue.get_nowait()
                    if delivery_time <= self.t_now:
                        # count the spike
                        with counts_lock:
                            spike_counts[self.nid] += 1
                    else:
                        # not yet – put it back and stop draining
                        self.spike_queue.put((delivery_time,))
                        break
            except queue.Empty:
                pass

            # ---- tiny sleep to avoid a busy spin ------------------------------------
            time.sleep(0.0001)        # 100 µs


# ----------------------------------------------------------------------
# Build the whole network and run the simulation
# ----------------------------------------------------------------------
def main(N_NEURONS=N_NEURONS, verbose=True):
    # --------------------------------------------------------------
    # 1️⃣  Create neurons (empty connection list for now)
    # --------------------------------------------------------------
    neurons = [Neuron(i, []) for i in range(N_NEURONS)]

    # --------------------------------------------------------------
    # 2️⃣  Freeze random connectivity + per‑link delays
    # --------------------------------------------------------------
    for i, n in enumerate(neurons):
        # Approx. 10 % outgoing degree (average ~10 targets)
        out_deg = max(1,
                      int(random.gauss(N_NEURONS * 0.1,
                                      N_NEURONS * 0.05)))
        # pick distinct targets, never self‑connect
        possible = list(range(N_NEURONS))
        possible.remove(i)
        targets = random.sample(possible,
                                min(out_deg, N_NEURONS - 1))

        for tgt_idx in targets:
            delay_s = random.uniform(MIN_DELAY_MS, MAX_DELAY_MS) / 1000.0
            n.connections.append((neurons[tgt_idx], delay_s))

    # --------------------------------------------------------------
    # 3️⃣  Start all neuron threads
    # --------------------------------------------------------------
    for n in neurons:
        n.start()

    # --------------------------------------------------------------
    # 4️⃣  Let the simulation run for the requested wall‑clock time
    # --------------------------------------------------------------
    time.sleep(SIM_TIME)
    stop_event.set()               # tell everybody to stop

    # --------------------------------------------------------------
    # 5️⃣  Wait for all threads to finish
    # --------------------------------------------------------------
    for n in neurons:
        n.join()

    # --------------------------------------------------------------
    # 6️⃣  Print a concise report (spike count per neuron)
    # --------------------------------------------------------------
    if verbose:
        print("\n=== Spike‑count report ===")
        total = sum(spike_counts.values())
        print(f"Total spikes received (all neurons) : {total}")
        for nid in range(N_NEURONS):
            cnt = spike_counts[nid]
            print(f"Neuron {nid:03d} : {cnt} spikes")

if __name__ == "__main__":
    main()

In [ ]:
for N_NEURONS_ in 2**np.arange(5, 14, dtype=int):
    tic = time.time()
    main(N_NEURONS=N_NEURONS_, verbose=False)
    toc = time.time()
    print(f'N_NEURONS={N_NEURONS_} , CPU wall time : {toc-tic:.2f}')

## using 0mq



In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
ZeroMQ + threading spiking‑network – robust version (fixed LINGER bug)

* A fresh ZeroMQ context is created for each experiment.
* The dispatcher receives the event, the priority queue and the lock dict
  as *explicit arguments* – no hidden globals.
* Each neuron’s PUB socket has `socket.linger = 0`; we do **not** call
  `ctx.set(zmq.LINGER, …)` because LINGER is a socket‑only option.
"""

import threading
import time
import random
import queue
import itertools
import zmq
from collections import defaultdict

# ----------------------------------------------------------------------
# Simulation parameters (feel free to tweak)
# ----------------------------------------------------------------------
MEAN_RATE_HZ    = 1.0
MIN_DELAY_MS    = 2
MAX_DELAY_MS    = 40
AVG_OUT_DEGREE  = 10
SIM_TIME_SEC    = 5.0

# ----------------------------------------------------------------------
# Helper – fresh ZeroMQ context (no LINGER set on the context)
# ----------------------------------------------------------------------
def make_context() -> zmq.Context:
    """Create a new ZeroMQ context.  LINGER is a *socket* option → set later."""
    return zmq.Context()

# ----------------------------------------------------------------------
# Dispatcher – the only code that really calls socket.send()
# ----------------------------------------------------------------------
def dispatcher(stop_evt: threading.Event,
              delayed_q: queue.PriorityQueue,
              pub_locks: dict[int, threading.Lock]) -> None:
    while not stop_evt.is_set() or not delayed_q.empty():
        try:
            deliver_time, _, src_pub = delayed_q.get(timeout=0.1)
        except queue.Empty:
            continue

        now = time.monotonic()
        if deliver_time > now:
            time.sleep(deliver_time - now)

        # Use a lock per PUB socket – ZeroMQ PUB sockets are thread‑safe,
        # but the lock removes any doubt.
        with pub_locks[id(src_pub)]:
            try:
                src_pub.send(b'\x00')
            except zmq.ZMQError as e:
                if e.errno != zmq.ENOTSOCK:
                    raise

        delayed_q.task_done()


# ----------------------------------------------------------------------
# Neuron – one thread per neuron
# ----------------------------------------------------------------------
class Neuron(threading.Thread):
    def __init__(self, nid: int,
                 out_links: list[tuple[int, float]],
                 ctx: zmq.Context):
        super().__init__(daemon=True)
        self.nid = nid
        self.out_links = out_links            # list of (target_id, delay_s)

        # ----- PUB socket (outgoing) -----------------------------------
        self.pub_socket = ctx.socket(zmq.PUB)
        self.pub_socket.linger = 0
        port = self.pub_socket.bind_to_random_port("tcp://127.0.0.1")
        self.pub_addr = f"tcp://127.0.0.1:{port}"

        # ----- SUB socket (incoming) -----------------------------------
        self.sub_socket = ctx.socket(zmq.SUB)
        self.sub_socket.linger = 0
        self.sub_socket.setsockopt(zmq.SUBSCRIBE, b"")

    # ------------------------------------------------------------------
    # After all neurons exist we connect our SUB socket to any source that
    # lists us as a target.
    # ------------------------------------------------------------------
    def connect_presynaptic(self, all_neurons: list["Neuron"]):
        for src in all_neurons:
            if src.nid == self.nid:
                continue
            for tgt_id, _ in src.out_links:
                if tgt_id == self.nid:
                    self.sub_socket.connect(src.pub_addr)
                    break   # one connection per source is enough

    # ------------------------------------------------------------------
    # Main loop – Poisson spikes + receiving
    # ------------------------------------------------------------------
    def run(self):
        next_spike = time.monotonic() + random.expovariate(MEAN_RATE_HZ)

        poller = zmq.Poller()
        poller.register(self.sub_socket, zmq.POLLIN)

        # Small warm‑up so PUB‑SUB handshake finishes
        time.sleep(0.1)

        while not STOP_EVENT.is_set():
            now = time.monotonic()

            # --------------------------------------------------------------
            # 1️⃣  Poisson spike generation
            # --------------------------------------------------------------
            if now >= next_spike:
                for tgt_id, delay_s in self.out_links:
                    delivery = now + delay_s
                    delayed_queue.put((delivery,
                                      next(_counter),
                                      self.pub_socket))
                next_spike = now + random.expovariate(MEAN_RATE_HZ)

            # --------------------------------------------------------------
            # 2️⃣  Receive incoming spikes
            # --------------------------------------------------------------
            if poller.poll(0):
                try:
                    while True:
                        self.sub_socket.recv(flags=zmq.NOBLOCK)
                        with counts_lock:
                            spike_counts[self.nid] += 1
                except zmq.Again:
                    pass   # nothing more to read

            # --------------------------------------------------------------
            # 3️⃣  Tiny sleep to avoid a hot spin‑loop
            # --------------------------------------------------------------
            time.sleep(0.00001)

        # ------------------------------------------------------------------
        # Clean‑up sockets
        # ------------------------------------------------------------------
        self.pub_socket.close(linger=0)
        self.sub_socket.close(linger=0)


# ----------------------------------------------------------------------
# Build the whole network (connectivity, sockets, threads)
# ----------------------------------------------------------------------
def build_network(N_NEURONS: int, ctx: zmq.Context) -> list[Neuron]:
    # 1️⃣  Create all neurons (no connections yet)
    neurons = [Neuron(nid, [], ctx) for nid in range(N_NEURONS)]

    # 2️⃣  Random frozen connectivity (now we have real neuron objects)
    for nid, neuron in enumerate(neurons):
        out_deg = max(1,
                      int(random.gauss(AVG_OUT_DEGREE,
                                      AVG_OUT_DEGREE * 0.4)))
        candidates = list(range(N_NEURONS))
        candidates.remove(nid)                     # no self‑loops
        targets = random.sample(candidates,
                                min(out_deg, N_NEURONS - 1))
        neuron.out_links = [
            (tgt,
             random.uniform(MIN_DELAY_MS, MAX_DELAY_MS) / 1000.0)
            for tgt in targets
        ]

    # 3️⃣  Wire the SUB sockets to all presynaptic PUBs
    for neuron in neurons:
        neuron.connect_presynaptic(neurons)

    # 4️⃣  Start every neuron thread
    for neuron in neurons:
        neuron.start()

    return neurons


# ----------------------------------------------------------------------
# Shut everything down cleanly (called after every experiment)
# ----------------------------------------------------------------------
def close_all(neurons: list[Neuron], ctx: zmq.Context) -> None:
    STOP_EVENT.set()
    for n in neurons:
        n.join()

    # Close any stray sockets (just to be safe)
    for n in neurons:
        try:
            n.pub_socket.close(linger=0)
        except Exception:
            pass
        try:
            n.sub_socket.close(linger=0)
        except Exception:
            pass

    # Finally, terminate the ZeroMQ context – this releases the TCP ports.
    ctx.term()


# ----------------------------------------------------------------------
# Run a single experiment (creates, runs, tears down)
# ----------------------------------------------------------------------
def run_one_experiment(N_NEURONS: int, verbose: bool = True):
    global ctx, delayed_queue, STOP_EVENT, spike_counts, counts_lock, pub_locks, _counter

    # ---- Fresh shared objects for THIS run ----------------------------
    ctx = make_context()
    delayed_queue = queue.PriorityQueue()
    STOP_EVENT = threading.Event()
    spike_counts = defaultdict(int)
    counts_lock = threading.Lock()
    pub_locks = defaultdict(threading.Lock)
    _counter = itertools.count()

    # ---- Dispatcher -------------------------------------------------
    disp_thread = threading.Thread(
        target=dispatcher,
        args=(STOP_EVENT, delayed_queue, pub_locks),
        daemon=True,
    )
    disp_thread.start()

    # ---- Build and start the network ---------------------------------
    neurons = build_network(N_NEURONS, ctx)

    # ---- Let it run -------------------------------------------------
    time.sleep(SIM_TIME_SEC)

    # ---- Clean‑up ---------------------------------------------------
    close_all(neurons, ctx)          # stops neurons & terminates context
    disp_thread.join()

    # ---- Report -----------------------------------------------------
    if verbose:
        total = sum(spike_counts.values())
        print("\n=== Spike‑count report ===")
        print(f"N_NEURONS = {N_NEURONS}")
        print(f"Total spikes received (all neurons): {total}")
        print(f"Average spikes per neuron          : {total/N_NEURONS:.2f}\n")
        for nid in range(min(5, N_NEURONS)):
            print(f"Neuron {nid:02d} received {spike_counts.get(nid,0)} spikes")
    return dict(spike_counts)


# ----------------------------------------------------------------------
# Example scaling loop (the one that raised the error before)
# ----------------------------------------------------------------------
if __name__ == "__main__":
    import numpy as np

    for N_NEURONS_ in 2 ** np.arange(5, 14, dtype=int):   # 32, 64, … 8192
        tic = time.time()
        run_one_experiment(N_NEURONS=N_NEURONS_, verbose=False)
        toc = time.time()
        print(f"N_NEURONS={N_NEURONS_:5d}  →  wall‑time: {toc - tic:.2f}s")

## ZeroMQ + multiprocessing spiking‑neuron network


In [12]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
ZeroMQ + multiprocessing spiking‑neuron network (process‑per‑neuron).

* One OS process per neuron, each with its own PUB/SUB sockets.
* Random frozen connectivity, per‑edge delay 2 ms – 40 ms.
* Poisson spike generation (λ = 1 Hz).
* Payload = single byte (sender id); receiver only counts spikes from its
  presynaptic partners.
* All functions/classes are defined at top‑level so that the “spawn” start‑method
  can import them on macOS / Windows.
"""

import multiprocessing as mp
import threading
import time
import random
import zmq
from collections import defaultdict

# ----------------------------------------------------------------------
# Simulation parameters (feel free to change)
# ----------------------------------------------------------------------
N_NEURONS        = 100               # number of neurons (processes)
AVG_OUT_DEGREE   = 10                # average outgoing connections per neuron
MIN_DELAY_MS     = 2                 # minimal axonal delay (ms)
MAX_DELAY_MS     = 40                # maximal axonal delay (ms)
MEAN_RATE_HZ     = 1.0               # Poisson firing rate (spikes / s)
SIM_TIME_SEC     = 5.0               # wall‑clock simulation length
BASE_PORT        = 5600              # PUB sockets listen on BASE_PORT + neuron_id


# ----------------------------------------------------------------------
# Helper – fresh ZeroMQ context (created **inside** each process)
# ----------------------------------------------------------------------
def make_context() -> zmq.Context:
    """Create a new ZeroMQ context.  LINGER is a socket option, not a context option."""
    return zmq.Context()


# ----------------------------------------------------------------------
# The function that runs in **each neuron process**
# ----------------------------------------------------------------------
def neuron_process(
    nid: int,
    out_links: list[tuple[int, float]],
    presynaptic: set[int],
    stop_evt: mp.Event,
    count_dict: dict,
):
    """
    One neuron lives in its own OS process.

    Parameters
    ----------
    nid           : integer, neuron identifier (0 … N_NEURONS‑1)
    out_links     : [(target_id, delay_seconds), …] – outgoing axons
    presynaptic   : set of neuron ids that send spikes to *this* neuron
    stop_evt      : multiprocessing.Event used to stop the loop
    count_dict    : multiprocessing‑managed dict where the final count is stored
    """
    # --------------------------------------------------------------
    # ZeroMQ context (one per process)
    # --------------------------------------------------------------
    ctx = make_context()

    # --------------------------------------------------------------
    # PUB socket – bind to a *deterministic* port (BASE_PORT + nid)
    # --------------------------------------------------------------
    pub = ctx.socket(zmq.PUB)
    pub.linger = 0
    bind_addr = f"tcp://127.0.0.1:{BASE_PORT + nid}"
    pub.bind(bind_addr)               # deterministic address – easier to debug
    my_addr = bind_addr               # keep for possible debugging

    # --------------------------------------------------------------
    # SUB socket – subscribe to everything
    # --------------------------------------------------------------
    sub = ctx.socket(zmq.SUB)
    sub.linger = 0
    sub.setsockopt(zmq.SUBSCRIBE, b"")

    # --------------------------------------------------------------
    # Connect SUB socket to every presynaptic PUB
    # --------------------------------------------------------------
    for src_id in presynaptic:
        src_addr = f"tcp://127.0.0.1:{BASE_PORT + src_id}"
        sub.connect(src_addr)

    # --------------------------------------------------------------
    # Warm‑up pause – give ZeroMQ time for its internal handshake
    # --------------------------------------------------------------
    time.sleep(0.1)

    # --------------------------------------------------------------
    # Helper that performs the *per‑link* axonal delay before publishing.
    # --------------------------------------------------------------
    def delayed_send(delay_s: float):
        """Sleep <delay_s> then publish a one‑byte payload (our nid)."""
        time.sleep(delay_s)
        pub.send(bytes([nid]))        # payload = sender id (one byte)

    # --------------------------------------------------------------
    # Main loop – generate Poisson spikes & receive inbound spikes
    # --------------------------------------------------------------
    next_spike = time.monotonic() + random.expovariate(MEAN_RATE_HZ)
    my_counter = 0                     # local spike count

    poller = zmq.Poller()
    poller.register(sub, zmq.POLLIN)

    while not stop_evt.is_set():
        now = time.monotonic()

        # --------------------------------------------------------------
        # 1️⃣  Poisson spike generation
        # --------------------------------------------------------------
        if now >= next_spike:
            for tgt_id, delay_s in out_links:
                threading.Thread(
                    target=delayed_send,
                    args=(delay_s,),
                    daemon=True,
                ).start()
            next_spike = now + random.expovariate(MEAN_RATE_HZ)

        # --------------------------------------------------------------
        # 2️⃣  Receive any spikes that have already arrived
        # --------------------------------------------------------------
        if poller.poll(0):                     # non‑blocking poll
            try:
                while True:
                    msg = sub.recv(flags=zmq.NOBLOCK)
                    sender_id = msg[0]        # one‑byte payload
                    if sender_id in presynaptic:
                        my_counter += 1
            except zmq.Again:
                pass   # queue empty

        # --------------------------------------------------------------
        # 3️⃣  Tiny sleep to avoid a hot spin‑loop
        # --------------------------------------------------------------
        time.sleep(0.00001)

    # ------------------------------------------------------------------
    # Clean‑up before exiting the process
    # ------------------------------------------------------------------
    pub.close(linger=0)
    sub.close(linger=0)
    ctx.term()

    # Store the final count in the shared dictionary (key = neuron id)
    count_dict[nid] = my_counter


# ----------------------------------------------------------------------
# Build the frozen random connectivity (parent process)
# ----------------------------------------------------------------------
def build_connectivity(num_neurons: int):
    """
    Returns two structures:
      * out_links[nid] = [(target_id, delay_seconds), …]
      * presynaptic[nid] = set of neuron ids that are presynaptic to nid
    """
    out_links = [[] for _ in range(num_neurons)]

    for nid in range(num_neurons):
        out_deg = max(
            1,
            int(random.gauss(AVG_OUT_DEGREE, AVG_OUT_DEGREE * 0.4)),
        )
        candidates = list(range(num_neurons))
        candidates.remove(nid)                     # no self‑loops
        targets = random.sample(
            candidates,
            min(out_deg, num_neurons - 1),
        )
        out_links[nid] = [
            (tgt,
             random.uniform(MIN_DELAY_MS, MAX_DELAY_MS) / 1000.0)
            for tgt in targets
        ]

    # Build inverse mapping (presynaptic set)
    presynaptic = [set() for _ in range(num_neurons)]
    for src_id, outs in enumerate(out_links):
        for tgt_id, _ in outs:
            presynaptic[tgt_id].add(src_id)

    return out_links, presynaptic


# ----------------------------------------------------------------------
# Parent – spawn processes, run simulation, collect results
# ----------------------------------------------------------------------
def run_experiment(num_neurons: int = N_NEURONS, verbose: bool = True):
    # ------------------------------------------------------------------
    # 0️⃣  Build connectivity once
    # ------------------------------------------------------------------
    out_links, presynaptic_sets = build_connectivity(num_neurons)

    # ------------------------------------------------------------------
    # 1️⃣  Shared multiprocessing objects
    # ------------------------------------------------------------------
    manager = mp.Manager()
    stop_evt = manager.Event()                 # will be set after SIM_TIME_SEC
    count_dict = manager.dict()                # neuron_id → spike count

    # ------------------------------------------------------------------
    # 2️⃣  Spawn one process per neuron
    # ------------------------------------------------------------------
    processes = []
    for nid in range(num_neurons):
        p = mp.Process(
            target=neuron_process,
            args=(
                nid,
                out_links[nid],          # list of (target_id, delay_s)
                presynaptic_sets[nid],  # set of presynaptic ids
                stop_evt,
                count_dict,
            ),
            name=f"Neuron-{nid}",
        )
        p.start()
        processes.append(p)

    # ------------------------------------------------------------------
    # 3️⃣  Let the simulation run for the requested wall‑clock time
    # ------------------------------------------------------------------
    time.sleep(SIM_TIME_SEC)

    # ------------------------------------------------------------------
    # 4️⃣  Signal everybody to stop and wait for clean termination
    # ------------------------------------------------------------------
    stop_evt.set()
    for p in processes:
        p.join()

    # ------------------------------------------------------------------
    # 5️⃣  Report
    # ------------------------------------------------------------------
    total = sum(count_dict.get(nid, 0) for nid in range(num_neurons))
    if verbose:
        print("\n=== Spike‑count report ===")
        print(f"N_NEURONS = {num_neurons}")
        print(f"Total spikes received (all neurons): {total}")
        print(f"Average spikes per neuron          : {total / num_neurons:.2f}\n")
        for nid in range(min(5, num_neurons)):
            print(f"Neuron {nid:02d} received {count_dict.get(nid,0)} spikes")

    # Return the dict for programmatic use (e.g. benchmarking)
    return dict(count_dict)


# ----------------------------------------------------------------------
# Example scaling loop (the one you tried before)
# ----------------------------------------------------------------------
if __name__ == "__main__":
    import numpy as np

    for N_NEURONS_ in 2 ** np.arange(5, 10, dtype=int):   # 32, 64, 128, 256, 512
        tic = time.time()
        run_experiment(N_NEURONS=N_NEURONS_, verbose=False)
        toc = time.time()
        print(f"N_NEURONS={N_NEURONS_:5d}  →  wall‑time: {toc - tic:.2f}s")

TypeError: run_experiment() got an unexpected keyword argument 'N_NEURONS'